# **SAD2, lab8:** Expectation-Maximization

*Justyna Król*



## **Expectation–Maximization (EM) Algorithm**

In many real-world problems we want to estimate model parameters even though part of the data is hidden or unobserved. Examples include clustering, mixture models, gene annotation, and Hidden Markov Models (HMMs). In such settings, the likelihood function becomes difficult to optimize directly because we must marginalize over all possible latent variable configurations.

Let:

$X$ = observed data

$Z$ = latent (unobserved) variables

$\theta$ = model parameters we want to estimate

We want to maximize the log-likelihood:

$$ \log p(X \mid \theta) = \log \sum_{Z} p(X, Z \mid \theta) $$

Direct maximization is often intractable because of the summation over $Z$.
EM solves this using two alternating steps:

#### **1) E-Step (Expectation)**

Compute $ Q(\theta \mid \theta^{(t)})$ defined as the expected value of the log likelihood function of  $\theta$, with respect to the current conditional distribution of  $Z$ given  $X$ and the current estimates of the parameters $\theta^{(t)}$:

$$
Q(\theta \mid \theta^{(t)})
= \mathbb{E}_{Z \sim p(\cdot \mid X, \theta^{(t)})} \left[ \log p(X, Z \mid \theta) \right]
:= \int \log p(X, Z \mid \theta) \, p(Z \mid X, \theta^{(t)}) \, dZ .
$$


#### **2) M-Step (Maximization)**

Update parameters by maximizing the $Q$ function:

$$ \theta^{(t+1)} = \arg\max_{\theta} Q(\theta \mid \theta^{(t)}) $$

This gives us the new parameters - then the EM cycle repeats until convergence.


## Reminder: Hidden Markov Models

>*Definition.*
A **Hidden Markov Model** $\mathcal{M}$ is a triplet  
$$
\mathcal{M} = (\Sigma, Q, \Theta),
$$
where:

- $\Sigma$ is an alphabet of observable symbols,  
- $Q$ is a finite set of hidden states,  
- $\Theta = (\pi, T, E)$ is a collection of probability distributions consisting of:  

  - **initial state probabilities** $\pi_i$ for each $i \in Q$:
  $$ \pi_i = \mathbb{P}(Z_1 = i) $$

  - **transition probabilities** $t_{i,j}$ for $i, j \in Q$:
  $$ t_{i,j} = \mathbb{P}(Z_n = j \mid Z_{n-1} = i) $$

  - **emission probabilities** $e_j(s)$ for $j \in Q$ and $s \in \Sigma$:
  $$ e_j(s) = \mathbb{P}(X_n = s \mid Z_n = j) $$

Here,  
- $X = (X_1, X_2, \ldots)$ is the sequence of observable symbols over $\Sigma$, and  
- $Z = (Z_1, Z_2, \ldots)$ is the sequence of hidden states over $Q$.


## EM for HMMs

**Motivation:**  
If the parameters $\theta$ of an HMM are known, the distribution over the latent states $Z$ can be computed efficiently using the **forward–backward algorithm**.  
Conversely, if the latent states $Z$ were known, estimating the parameters $\theta$ would be straightforward: we could count how often transitions and emissions occur and normalize these counts to obtain updated transition, emission, and initial probabilities.

This motivates an iterative procedure when both $\theta$ and $Z$ are unknown:

1. Initialize the parameters $\theta$.
2. **E-step:** Using the forward–backward algorithm, compute the posterior distribution of the latent states $Z$ given the current parameters $\theta$.
3. **M-step:** Update $\theta$ using the expected sufficient statistics of $Z$ computed in the E-step.
4. Repeat steps 2–3 until convergence.

This EM procedure guarantees a monotonic increase in the data log-likelihood and converges to a local optimum.


1. Krok jest potrzebny do znalezienia prd. emisji, oraz poczatkowych wartości
2. Krok jest potrzebny do znalezienia macierzy przejścia (jak to się dokładnie wyprowadza)

Ważne
- tutaj trzeba pokazac jak to zapisać za pomoća pozostałych algorytmów

## **Baum–Welch Algorithm**

The **Baum–Welch algorithm** is an EM procedure for HMMs that iteratively updates parameters $\theta = (\pi, T, E)$ when the latent states $Z$ are unknown.

**Steps:**

1. **E-step:** Compute the posterior probabilities of hidden states using the **forward–backward algorithm**:

$$
\gamma_n(i) = \mathbb{P}(Z_n = i \mid X, \theta), $$
$$
\xi_n(i,j) = \mathbb{P}(Z_n = i, Z_{n+1} = j \mid X, \theta)
$$

2. **M-step:** Update the parameters using these expected counts:

- Initial state probabilities:
$$
\pi_i^{\text{new}} = \gamma_1(i)
$$

- Transition probabilities:
$$
t_{ij}^{\text{new}} = \frac{\sum_{n=1}^{N-1} \xi_n(i,j)}{\mathbb{P}(X)}
$$

- Emission probabilities:
$$
e_j(s)^{\text{new}} = \frac{\sum_{n=1}^{N} \gamma_n(j) \mathbf{1}_{\{X_n = s\}}}{\mathbb{P}(X)}
$$

Repeat E-step and M-step until convergence. The algorithm monotonically increases the data log-likelihood and converges to a local optimum.


## **Theoretical task 1:**

Let $X_1, \dots, X_N$ be a sequence of observations in an HMM.  
Given:  

- the current parameter estimates $\theta = (\pi, T, E)$,  
- the output of the forward function $f_{i, n} = \mathbb{P}(X_{1:n}, Z_n = i )$,  
- the output of the backward function $b_{i, n} = \mathbb{P}(X_{n+1:N} \mid Z_n = i)$,  

prove that the joint posterior of consecutive hidden states is

$$
\xi_n(i,j) = \mathbb{P}(Z_n = i, Z_{n+1} = j \mid X) =  \frac{f_{i,n} \, t_{ij} \, e_j(X_{n+1}) \, b_{j,n+1}}{\mathbb{P}(X)}
$$




### Using Log Probabilities in Forward, Backward, and Baum–Welch Algorithms

In this lab, we will work with **log probabilities** to improve numerical stability when dealing with long sequences. Accordingly, the **forward, backward, and Baum–Welch algorithms** will be implemented in a log-space version.  

- In the **forward algorithm**, we compute the log of the forward variables $\log f_{i,n}$, replacing products of probabilities with sums of logs.  
- Similarly, the **backward algorithm** uses $\log b_{i,n}$ to propagate probabilities backward in log-space.  
- The **Baum–Welch EM updates** are also computed using log probabilities, ensuring that expected counts and parameter updates remain numerically stable.  

Using log-space computations prevents underflow, allows handling long sequences, and ensures that all steps of the EM procedure are robust.


### **Exercises**

You are provided with a working implementation of the **Baum–Welch algorithm** for Hidden Markov Models. Your goal is to examine how the algorithm's convergence behavior depends on **initialization**, **sequence length**, and **number of iterations**.

First, briefly the code below.



In [ ]:
import torch
import numpy as np
import matplotlib.pyplot as plt

def set_seed(seed):
    torch.manual_seed(seed)
    np.random.seed(seed)

def forward_log(T_log, E_log, pi_log, sequence):
    """
    Forward algorithm in log space.
    T_log: log transition matrix (K x K)
    E_log: log emission matrix  (K x M)
    pi_log: log initial probabilities (K)
    sequence: observed sequence (list or tensor of ints)
    """
    K = E_log.shape[0]
    L = len(sequence)

    f_log = torch.zeros((K, L), dtype=torch.float32)

    # Initialization
    f_log[:, 0] = pi_log + E_log[:, sequence[0]]

    # Recursion
    for i in range(1, L):
        for l in range(K):
            f_log[l, i] = E_log[l, sequence[i]] + torch.logsumexp(f_log[:, i-1] + T_log[:, l], dim=0)

    loglik = torch.logsumexp(f_log[:, -1], dim=0)

    return f_log, loglik


def backward_log(T_log, E_log, sequence):
    """
    Backward algorithm in log space.
    T_log: log transition matrix (K x K)
    E_log: log emission matrix (K x M)
    sequence: observed sequence
    """
    K = T_log.shape[0]
    L = len(sequence)

    b_log = torch.zeros((K, L), dtype=torch.float32)

    # Initialization
    b_log[:, L - 1] = 0.

    for i in range(L - 2, -1, -1):
        for k in range(K):
            b_log[k, i] = torch.logsumexp(T_log[k, :] + E_log[:, sequence[i + 1]] + b_log[:, i + 1], dim=0)

    return b_log


def get_posterior(T_log, E_log, pi_log, sequence):
    """
    Compute posterior p(z_i | x) from log forward and log backward scores
    Posterior is returned in ordinary prob space.
    """
    f_log, loglik = forward_log(T_log, E_log, pi_log, sequence)
    b_log = backward_log(T_log, E_log, sequence)

    log_posterior = f_log + b_log

    # Convert to normalized probabilities
    log_posterior -= torch.logsumexp(log_posterior, dim=0, keepdim=True)

    posterior = torch.exp(log_posterior)
    return posterior


def baum_welch_log(sequences, K, M, init_T, init_E, init_pi, max_iter=20, tol=1e-4, num_restarts=10):
    """
    Baum–Welch training for multiple sequences using log-space forward/backward.

    sequences: list of int tensors (each length L_n)
    K: number of hidden states
    M: number of emission symbols
    """
    T_log = torch.log(init_T + 1e-12)
    E_log = torch.log(init_E + 1e-12)
    pi_log = torch.log(init_pi + 1e-12)

    last_loglik = -float("inf")
    logliks = []
    for iteration in range(max_iter):
        # Accumulators for expected counts
        gamma_sum_for_transitions = torch.zeros((K,))
        gamma_init_sum = torch.zeros((K,))
        emission_sum = torch.zeros((K, M))
        xi_sum = torch.zeros((K, K))

        total_loglik = 0.0

        # E-step: calculate log_likelihood
        for seq in sequences:
            # Forward / Backward in log space
            f_log, loglik = forward_log(T_log, E_log, pi_log, seq)
            b_log = backward_log(T_log, E_log, seq)

            total_loglik += loglik.item()

            # Posterior state probabilities
            gamma_log = f_log + b_log - loglik
            gamma = torch.exp(gamma_log)         # (K × N)

            # Posterior transition probabilities: xi
            N = len(seq)
            for i in range(N - 1):
                xi_log = (
                    f_log[:, i].unsqueeze(1)
                    + T_log
                    + E_log[:, seq[i + 1]].unsqueeze(0)
                    + b_log[:, i + 1].unsqueeze(0)
                    - loglik
                )
                xi = torch.exp(xi_log)  # (K × K)
                xi_sum += xi

            # Accumulate expected counts
            gamma_sum_for_transitions += torch.sum(gamma[:, :-1], dim=1)
            gamma_init_sum += gamma[:, 0]
            for i, x in enumerate(seq):
                emission_sum[:, x] += gamma[:, i]

        # M-step: update probabilities
        pi = gamma_init_sum / torch.sum(gamma_init_sum)
        T = xi_sum / gamma_sum_for_transitions.unsqueeze(1)
        E = emission_sum / torch.sum(emission_sum, dim=1, keepdim=True)

        # Convert to log-space for next iteration
        T_log = torch.log(T + 1e-12)
        E_log = torch.log(E + 1e-12)
        pi_log = torch.log(pi + 1e-12)

        # Check convergence
        if iteration>5 and np.abs(total_loglik - last_loglik) < tol:
            print(f"Converged after {iteration + 1} iterations.")
            break
        last_loglik = total_loglik
        logliks.append(total_loglik)
        print(f"Iter {iteration + 1}: log-likelihood = {total_loglik:.4f}")
        print("T: ", torch.round(T, decimals=4))
        print("E: ", torch.round(E, decimals=4))
        print("pi: ", torch.round(pi, decimals=4))


    return T, E, pi, logliks

## **Exercise 1: Investigating Baum–Welch Convergence**

1. **Generate observation sequences**  
- Use a known HMM with specified parameters $(\pi, T, E)$.
$$
  \pi = [0.5, 0.5], \quad
  T = \begin{bmatrix} 0.9 & 0.1 \\ 0.2 & 0.8 \end{bmatrix}, \quad
  E = \begin{bmatrix} 0.5 & 0.5 \\ 0.1 & 0.9 \end{bmatrix}
$$
- Generate a set of 5 observation sequences from this model, each of length 500.

Hint: use ``torch.multinomial``

2. **Examine the impact of initialization**  
  - Run the Baum–Welch algorithm multiple times using
    - 15 different random initial parameter values.  
    - uniform initialization, e.g., all transition and emission probabilities equal and rows summing to 1:  
$$
    \pi = [0.5, 0.5], \quad
    T = \begin{bmatrix} 0.5 & 0.5 \\ 0.5 & 0.5 \end{bmatrix}, \quad
    E = \begin{bmatrix} 0.5 & 0.5 \\ 0.5 & 0.5 \end{bmatrix}
$$
  - Create a line plot that shows log-likelihood over Baum–Welch iterations

    - each run should appear as its own line

    - x-axis: iteration number

    - y-axis: log-likelihood

    Include a reference line showing the log-likelihood of the data under the true HMM parameters used for generation.

**Questions**:

What effects do different initializations have on convergence?

How does final convergence relate to the initial log-likelihood?

Do you observe special cases such as plateaus, rapid jumps, or local optima?

Is uniform initialization effective or not? Why?

Can you propose a better parameter initialization strategy?

In [ ]:
# tylko baum-welsha używamy 

# parameters
import stat


PI_TRUE = torch.tensor([0.5, 0.5])
T_TRUE = torch.tensor([[0.9, 0.1], [0.2, 0.8]])
E_TRUE = torch.tensor([[0.5, 0.5], [0.1, 0.9]])
sequence_length = 500

# generating observations (keep all data - for conclusions)
samples = []
hidden_states = []
for _ in range(5):
    # initalization of samples
    observation_sequence = torch.empty(sequence_length, dtype=torch.int16)
    state_sequence = torch.empty(sequence_length, dtype=torch.int16)
    # sample init state
    state_sequence[0] = torch.multinomial(PI_TRUE, 1).item() 
    observation_sequence[0] = torch.multinomial(E_TRUE[state_sequence[0], :], 1).item()
    # rest states 
    for i in range(1, sequence_length):
        state_sequence[i] = torch.multinomial(T_TRUE[state_sequence[i-1], :], 1).item()
        observation_sequence[i] = torch.multinomial(E_TRUE[state_sequence[i], :], 1).item()
    # appending probe
    samples.append(observation_sequence)    
    hidden_states.append(state_sequence)
    # TODO - to można efektywniej, przestawiając indeksy

Ważne
- przy jednostajnym rozkładzie, możemy bardzo łatwo przeskakiwać między stanami i okazuje się że model prewiduje średnią wartość oczekiwaną wystąpienia stanów 
- bardzo ważna jest inicjalizacja, trzeba mniej więcej przejść przez wszystkie stany, żeby otrzymać
- overfitting i underfitting
    - tutaj trzeba opisać dokładniej metody weryfikacji tego.
    - ciężko wychwycić podobne prawdopodobieńśtwa, ponieważ nie można stwierdzic w jakim stanie się jest 
- porównywanie modeli
    - ważne przejrzenie wynikówk, ponieważ model nie rozróżnia stanów - musimy im interpretacje przypisać 
    - porównujemy pradopodobieństwo otrzymania (bo samych stanów on i tka nie rozróżni, więc możemy porównać tylko problem który mu zadaliśmy) 

In [ ]:
T

In [ ]:
# initial condition 
PI_INIT = torch.tensor([0.5, 0.5])
T_INIT = torch.tensor([[0.5, 0.5], [0.5, 0.5]])
E_INIT = torch.t_copy(T_INIT)

T, E, pi, logliks = baum_welch_log(samples[1].reshape(-1, 1), T_INIT.shape[0], E_INIT.shape[0], T_INIT, E_INIT, PI_INIT)


In [ ]:
# analysis 
# plot log-likelihood vs Baum-Welch iterations 
plt.plot(logliks, )

### **Exercise 2: Posterior state probabilities and parameter evaluation**

1. **Generate a new sequence**  
   - Using the **true model parameters** $(\pi, T, E)$, generate a new short (length = 50) observation sequence $X_{\text{new}}$ and record the corresponding hidden states $Z_{\text{true}}$.

2. **Select learned parameters**  
   - From your Baum–Welch runs, choose the parameter set $(\hat{\pi}, \hat{T}, \hat{E})$ that achieved the **highest log-likelihood** on the training sequence.

3. **Compute posterior state probabilities**  
   - Using the selected learned parameters $(\hat{\pi}, \hat{T}, \hat{E})$, run the **forward–backward algorithm** to compute the posterior probabilities $\gamma_n(i) = \mathbb{P}(Z_n = i \mid X_{\text{new}}, \hat{\theta})$ for each time step.  
   - Repeat the computation using the **true parameters** $(\pi, T, E)$ for comparison.

4. **Plot the posteriors**
   - Plot the **posterior probabilities over time** computed from learned vs. true parameters for each hidden state.  
   - Overlay the **true hidden states** to visually assess how well the learned model captures the latent sequence.  

**Questions:**

How close are the posterior probabilities from the learned parameters to those from the true parameters?

Does the learned model recover the hidden states accurately?

Can you identify sequences or time steps where the model struggles?

5. **Repeat steps 3-4 for different learned parameters.** How do posteriors change? Is loglikelihood a good predictor of how well the leared model performs?


### Additional exercises:

1. Repeat the analysis for a bigger model, eg. with 4 hidden states and 6 possible emissions. Experiment with the size of data used for training. What are your observations?

2. Partial parameter learning
   - Assume the **emission matrix is known**. Modify the Baum–Welch algorithm to update only the **transition matrix** and **initial state probabilities**.  
   - Investigate convergence behavior in this scenario and compare to learning all parameters simultaneously.  
   - Discuss whether convergence is faster or more stable when fewer parameters are estimated.
